# 01 — Veri Katmanı (Data Layer)

**Cold-Start Sosyal-Grafik Sızıntısı — Simülasyon Çalışması**

Bu notebook gerçekçi sosyal ağ verisini üretir ve **Google Drive'a kaydeder.** Sonraki notebook (02 — Öneri Modeli) veriyi doğrudan Drive'dan okur. Böylece notebook'lar ayrı ayrı yüklenip sırayla çalıştırılabilir.

### Tasarım ilkesi: döngüselliği (circularity) kırmak
Sızıntıyı **biz tasarlamıyoruz.** Bu katman yalnızca *gerçekçi yapı* üretir: yönlü "takip" grafı (`u → v` = u, v'yi takip ediyor), `public/private` etiketler, ve topluluk yapısıyla **ayarlanabilir** korelasyona sahip düğüm özellikleri (meşru ilgi/coğrafya proxy'leri). Sızıntı ileride *standart, sızdırmak için tasarlanmamış* bir öneri sisteminden kendiliğinden çıkacak.

### Simülasyonun asıl kazancı: **ground-truth**
Hedefin gerçek komşuluğu `N(t) = takipçiler ∪ takip edilenler` burada tam bilinir → ileride precision **ve** recall raporlanabilir. Hiçbir canlı API, gerçek özel hesap veya ToS ihlali yok.


## 0. Kurulum ve Google Drive bağlama

In [ ]:
!pip install -q networkx numpy pandas scipy scikit-learn matplotlib

In [ ]:
import os, json, gzip, pickle, urllib.request
from dataclasses import dataclass, field, asdict
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# --- Google Drive'ı bağla (Colab) ---
def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/cold_start_leakage/data'
else:
    DATA_DIR = os.path.abspath('./cold_start_leakage/data')   # Colab dışı (yerel test)

os.makedirs(DATA_DIR, exist_ok=True)
print("Veriler buraya kaydedilecek:", DATA_DIR)
print("networkx", nx.__version__)

## 1. Konfigürasyon

Tüm "düğmeler" tek yerde. `homophily` = meşru sinyal gücü (0 = rastgele özellik, 1 = tam ayrışık); ileride ablasyonun temeli.


In [ ]:
@dataclass
class DataConfig:
    name: str = "sbm_default"
    seed: int = 42
    private_ratio: float = 0.30           # private (kapalı) hesap oranı
    private_mode: str = "random"          # "random" | "low_degree"
    mutual_ratio: float = 0.49            # karşılıklı takip olasılığı (~Table VI)
    feature_dim: int = 16
    homophily: float = 0.70               # meşru sinyal gücü
    # SBM
    sbm_sizes: tuple = (180, 160, 140, 120, 100)
    sbm_p_in: float = 0.06
    sbm_p_out: float = 0.0015
    # LFR
    lfr_n: int = 700
    lfr_tau1: float = 2.7
    lfr_tau2: float = 1.4
    lfr_mu: float = 0.18
    lfr_avg_degree: int = 12
    lfr_min_community: int = 60
    # gerçek graf (SNAP ego-Facebook)
    snap_url: str = "https://snap.stanford.edu/data/facebook_combined.txt.gz"

## 2. Standart veri seti konteyneri (şema)

Sonraki notebook bu alanlara güvenir: `G` (yönlü graf), `private`, `community`, `node_list`, `features`. Ground-truth yardımcıları: `followers / following / neighborhood`.


In [ ]:
@dataclass
class SocialGraphDataset:
    name: str
    G: nx.DiGraph                 # u -> v : "u, v'yi takip ediyor"
    private: dict
    community: dict
    node_list: list
    features: np.ndarray
    meta: dict = field(default_factory=dict)
    def followers(self, t):  return set(self.G.predecessors(t))   # t'yi takip edenler (in)
    def following(self, t):  return set(self.G.successors(t))     # t'nin takip ettikleri (out)
    def neighborhood(self, t): return self.followers(t) | self.following(t)   # N(t) — GROUND TRUTH

## 3. Yapı taşları: yönlendirme, etiketleme, özellikler

In [ ]:
def orient_edges(G_und, mutual_ratio, rng):
    # yönsüz taban -> yönlü takip grafı
    D = nx.DiGraph(); D.add_nodes_from(G_und.nodes())
    for u, v in G_und.edges():
        if rng.random() < mutual_ratio:
            D.add_edge(u, v); D.add_edge(v, u)            # karşılıklı
        else:
            if rng.random() < 0.5: D.add_edge(u, v)        # tek yönlü
            else:                  D.add_edge(v, u)
    return D

def assign_private(G, ratio, mode, rng):
    nodes = list(G.nodes()); n_priv = int(round(ratio * len(nodes)))
    if mode == "low_degree":
        deg = np.array([G.degree(n) for n in nodes], dtype=float)
        w = 1.0/(1.0+deg); w /= w.sum()
        chosen = rng.choice(len(nodes), size=n_priv, replace=False, p=w)
    else:
        chosen = rng.choice(len(nodes), size=n_priv, replace=False)
    priv = {n: False for n in nodes}
    for i in chosen: priv[nodes[i]] = True
    return priv

def make_features(node_list, community, dim, homophily, rng):
    # topluluk merkezi + gürültü; homophily arttıkça gürültü azalır (meşru sinyal güçlenir)
    comms = sorted(set(community.values()))
    centroids = rng.normal(size=(len(comms), dim))
    centroids /= (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-9)
    cidx = {c: i for i, c in enumerate(comms)}
    noise = (1.0 - homophily) + 1e-3
    X = np.zeros((len(node_list), dim))
    for i, n in enumerate(node_list):
        X[i] = centroids[cidx[community[n]]] + rng.normal(scale=noise, size=dim)
    return X

## 4. Graf üreteçleri (SBM, LFR, gerçek ego-Facebook)

In [ ]:
def gen_sbm(cfg, rng):
    sizes = list(cfg.sbm_sizes); k = len(sizes)
    p = [[cfg.sbm_p_in if i==j else cfg.sbm_p_out for j in range(k)] for i in range(k)]
    G = nx.stochastic_block_model(sizes, p, seed=int(rng.integers(1<<31)))
    return G, {n: G.nodes[n]["block"] for n in G.nodes()}

def gen_lfr(cfg, rng, max_tries=8):
    last = None
    for _ in range(max_tries):
        try:
            G = nx.LFR_benchmark_graph(n=cfg.lfr_n, tau1=cfg.lfr_tau1, tau2=cfg.lfr_tau2,
                                       mu=cfg.lfr_mu, average_degree=cfg.lfr_avg_degree,
                                       min_community=cfg.lfr_min_community, max_iters=1000,
                                       seed=int(rng.integers(1<<31)))
            G.remove_edges_from(nx.selfloop_edges(G))
            raw = {n: frozenset(G.nodes[n]["community"]) for n in G.nodes()}
            uniq = {fs: i for i, fs in enumerate(sorted(set(raw.values()), key=lambda s: min(s)))}
            return G, {n: uniq[raw[n]] for n in G.nodes()}
        except Exception as e:
            last = e
    raise RuntimeError(f"LFR {max_tries} denemede başarısız: {last}")

def load_facebook(cfg, rng):
    raw = urllib.request.urlopen(cfg.snap_url, timeout=60).read()
    G = nx.parse_edgelist(gzip.decompress(raw).decode().strip().split("\n"), nodetype=int)
    community = {}
    for ci, cset in enumerate(nx.community.greedy_modularity_communities(G)):
        for n in cset: community[n] = ci
    return G, community

## 5. Tek giriş noktası: `build_dataset`

In [ ]:
def build_dataset(cfg: DataConfig) -> SocialGraphDataset:
    rng = np.random.default_rng(cfg.seed)
    if   cfg.name.startswith("sbm"): G_und, community = gen_sbm(cfg, rng)
    elif cfg.name.startswith("lfr"): G_und, community = gen_lfr(cfg, rng)
    elif cfg.name.startswith(("facebook","real")): G_und, community = load_facebook(cfg, rng)
    else: raise ValueError(f"Bilinmeyen veri seti: {cfg.name}")

    G_und = nx.convert_node_labels_to_integers(G_und, label_attribute="orig")
    community = {i: community[G_und.nodes[i]["orig"]] for i in G_und.nodes()}

    D = orient_edges(G_und, cfg.mutual_ratio, rng)
    private = assign_private(D, cfg.private_ratio, cfg.private_mode, rng)
    node_list = sorted(D.nodes())
    features = make_features(node_list, community, cfg.feature_dim, cfg.homophily, rng)
    meta = {"config": asdict(cfg), "n_nodes": D.number_of_nodes(), "n_edges": D.number_of_edges()}
    return SocialGraphDataset(cfg.name, D, private, community, node_list, features, meta)

def summarize(ds: SocialGraphDataset) -> dict:
    D = ds.G; und = D.to_undirected(reciprocal=False)
    mutual = sum(1 for u, v in und.edges() if D.has_edge(u, v) and D.has_edge(v, u))
    try:
        from sklearn.metrics import silhouette_score
        labels = np.array([ds.community[n] for n in ds.node_list])
        sil = float(silhouette_score(ds.features, labels)) if len(set(labels)) > 1 else float("nan")
    except Exception:
        sil = float("nan")
    degs = [d for _, d in D.degree()]
    return {"n_nodes": D.number_of_nodes(), "n_directed_edges": D.number_of_edges(),
            "n_relations": und.number_of_edges(),
            "mutual_share": round(mutual/max(und.number_of_edges(),1), 3),
            "n_communities": len(set(ds.community.values())),
            "private_ratio": round(float(np.mean(list(ds.private.values()))), 3),
            "avg_degree": round(float(np.mean(degs)), 2), "max_degree": int(np.max(degs)),
            "feature_silhouette": round(sil, 3)}

## 6. Drive'a kaydet / Drive'dan oku

Her veri seti `DATA_DIR/<isim>/` altına yazılır: `edges.csv`, `nodes.csv`, `features.npy`, `graph.pkl`, `meta.json`. Bu klasörler Google Drive'da kalıcıdır; sonraki notebook `load_dataset(...)` ile doğrudan okur.


In [ ]:
def save_dataset(ds, root=DATA_DIR):
    outdir = os.path.join(root, ds.name); os.makedirs(outdir, exist_ok=True)
    nx.write_edgelist(ds.G, os.path.join(outdir, "edges.csv"), delimiter=",", data=False)
    pd.DataFrame({"node_id": ds.node_list,
                  "private": [ds.private[n] for n in ds.node_list],
                  "community": [ds.community[n] for n in ds.node_list]}
                 ).to_csv(os.path.join(outdir, "nodes.csv"), index=False)
    np.save(os.path.join(outdir, "features.npy"), ds.features)
    with open(os.path.join(outdir, "graph.pkl"), "wb") as f: pickle.dump(ds.G, f)
    with open(os.path.join(outdir, "meta.json"), "w") as f:
        json.dump({**ds.meta, "summary": summarize(ds)}, f, indent=2)
    return outdir

def load_dataset(name, root=DATA_DIR) -> SocialGraphDataset:
    indir = os.path.join(root, name)
    with open(os.path.join(indir, "graph.pkl"), "rb") as f: G = pickle.load(f)
    ndf = pd.read_csv(os.path.join(indir, "nodes.csv"))
    features = np.load(os.path.join(indir, "features.npy"))
    with open(os.path.join(indir, "meta.json")) as f: meta = json.load(f)
    return SocialGraphDataset(name, G, dict(zip(ndf.node_id, ndf["private"])),
                              dict(zip(ndf.node_id, ndf.community)), list(ndf.node_id), features, meta)

## 7. Veri setlerini üret ve Drive'a kaydet

In [ ]:
configs = [DataConfig(name="sbm_default"), DataConfig(name="lfr_default")]
rows = []
for cfg in configs:
    ds = build_dataset(cfg)
    out = save_dataset(ds)
    rows.append({"dataset": cfg.name, **summarize(ds)})
    print("kaydedildi ->", out)

pd.DataFrame(rows).set_index("dataset")

## 8. (İsteğe bağlı) Gerçek graf — SNAP ego-Facebook

In [ ]:
try:
    fb = build_dataset(DataConfig(name="facebook_real"))
    print("kaydedildi ->", save_dataset(fb))
    print(summarize(fb))
except Exception as e:
    print("İndirme atlandı/başarısız (sentetik setlerle devam edilebilir):", e)

## 9. Hızlı doğrulama — Drive'dan geri oku + ground-truth

In [ ]:
ds = load_dataset("sbm_default")     # sonraki notebook tam da bunu yapacak
targets = [n for n in ds.node_list if ds.private[n] and ds.G.degree(n) >= 4]
t = targets[0]
print("Drive'dan okundu:", ds.name)
print(f"Private hedef {t}: takipçi={len(ds.followers(t))}, takip={len(ds.following(t))}, "
      f"N(t)={len(ds.neighborhood(t))} | toplam aday private hedef={len(targets)}")

## Sonraki notebook için

**02 — Öneri Modeli** şu satırla başlayacak (aynı Drive yolu, aynı şema):

```python
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/cold_start_leakage/data'
ds = load_dataset("sbm_default", DATA_DIR)   # SocialGraphDataset + load_dataset tanımları tekrar gelir
```

Üzerinde standart (sızdırmak için tasarlanmamış) LightGCN / GraphSAGE / GraFRank tarzı modeller kurulacak; cold→warm gözlemci ekseni bağımsız değişken olacak. Modeller de Drive'a (`models/`) kaydedilecek.
